# 01 — Environment test, data build, leakage gate, real-model smoke test (Prompt 08 → Gate G2)

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **On**, Persistence **Files**.

Prompt 08 rule: this notebook **stops after the smoke test**. Download `results/env/` and `results/pilot/`
(or copy the printed outputs of cells 2, 3, 5, 6, 7, 9) and share them before anything else runs.
Nothing produced here is a research result.

## 1. Get the repository

In [ ]:
REPO_URL = "https://github.com/rovermask/AITR_Lab02_Project.git"   # change if your repo is elsewhere
ZIP_PATH = None   # or "/kaggle/input/<dataset>/AITR_Lab02_Project.zip" if uploaded as a Kaggle Dataset
import os, subprocess
os.chdir("/kaggle/working")
if ZIP_PATH:
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", "/kaggle/working"], check=True)
elif not os.path.exists("AITR_Lab02_Project"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("/kaggle/working/AITR_Lab02_Project")
os.makedirs("results/env", exist_ok=True)
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)

## 2. GPU / system diagnostics (raw)

In [ ]:
!nvidia-smi | tee results/env/nvidia_smi.txt
!python --version

## 3. Install dependencies and record exact versions
Versions are **not** pinned in advance: which vLLM build works with Kaggle's current CUDA/torch image is
[TEST REQUIRED]. Whatever passes cells 6–9 is frozen into `results/env/pip_freeze.txt` and then copied
into `requirements-gpu.txt`. If `pip install vllm` replaces Kaggle's torch and breaks CUDA, restart and
try the fallback in cell 3b.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets vllm 2>&1 | tail -5
!pip freeze > results/env/pip_freeze.txt
!grep -iE "^(vllm|torch|transformers|datasets|numpy|pandas|scipy)==" results/env/pip_freeze.txt

In [ ]:
# 3b. FALLBACK only if vLLM cannot import or cannot see the GPU: use Hugging Face transformers instead
# (much slower; run_experiment.py --backend hf). Leave commented unless needed.
# !pip install -q transformers accelerate

## 4. Raw MedQA test split (GitHub copy) — hash must match the value recorded on 2026-10-20

In [ ]:
import os, subprocess
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/medqa_test.jsonl"):
    subprocess.run(["git","clone","--depth","1","https://github.com/gersteinlab/MedAgents.git","/tmp/MedAgents"], check=True)
    subprocess.run(["cp","/tmp/MedAgents/datasets/MedQA/test.jsonl","data/raw/medqa_test.jsonl"], check=True)
!python scripts/prepare_data.py --src data/raw/medqa_test.jsonl --out data/processed

## 5. Environment test (expected: JSON, `problems` empty except notes; exit 0)

In [ ]:
!python scripts/env_check.py --require-gpu --json results/env/env_check_kaggle.json; echo "exit=$?"

## 6. Build MedMCQA oncology subset, MedQA 4-option anchor, pilot items (needs Hugging Face)

In [ ]:
!python scripts/prepare_extra_data.py --out data/processed | tee results/env/extra_manifest_print.txt

## 7. Leakage gate (both must print `n_flagged_pairs: 0` and exit 0)

In [ ]:
!python scripts/leakage_checks.py data/processed/medqa_train_pilot.jsonl data/processed/medqa_test_all.jsonl > results/env/leak_pilot_vs_test.json; echo "exit=$?"
!python scripts/leakage_checks.py data/processed/medmcqa_dev_onc.jsonl data/processed/medqa_test_onc.jsonl > results/env/leak_medmcqa_vs_medqa.json; echo "exit=$?"
!grep n_flagged results/env/leak_*.json

## 8. Unit tests (CPU, mock backend — checks the code, not the model)

In [ ]:
!python -m pytest -q tests | tail -3

## 9. Real-model smoke test on 5 MedQA **train** items (never reported)
Model `Qwen/Qwen2.5-7B-Instruct`, fp16, tensor-parallel over 2×T4 (protocol v1.1). If loading fails with
out-of-memory, follow the fallback in `configs/frozen_protocol.yaml` and log it.

In [ ]:
import time, json, subprocess
t=time.time()
subprocess.run(["python","scripts/run_experiment.py","--dataset","pilot_train","--method","cot",
                "--seed","0","--backend","vllm","--limit","5","--out","results/pilot"], check=True)
print(f"CoT x5 incl. model load: {time.time()-t:.0f}s")
for r in map(json.loads, open("results/pilot/pilot_train__cot__Qwen2.5-7B-Instruct__seed0.jsonl")):
    print("="*80); print("gold:", r["gold"], "| parsed:", r["final"], "| tokens:", r["prompt_tokens"], r["completion_tokens"])
    print(r["samples"][0]["text"][-700:])

In [ ]:
import time, subprocess, json
t=time.time()
subprocess.run(["python","scripts/run_experiment.py","--dataset","pilot_train","--method","board_roles",
                "--seed","0","--backend","vllm","--limit","3","--chunk","3","--out","results/pilot"], check=True)
print(f"board_roles x3 incl. model load: {time.time()-t:.0f}s")
for r in map(json.loads, open("results/pilot/pilot_train__board_roles__Qwen2.5-7B-Instruct__seed0.jsonl")):
    print(r["item_id"], "gold", r["gold"], "| R1", [m["answer"] for m in r["rounds"][0].values()],
          "| R2", [m["answer"] for m in r["rounds"][1].values()], "| chair", r["final"],
          "| api tokens", r["prompt_tokens"]+r["completion_tokens"])

In [ ]:
!cd results && zip -qr /kaggle/working/env_and_pilot_outputs.zip env pilot && ls -la /kaggle/working/env_and_pilot_outputs.zip

## STOP HERE (Prompt 08)
Share `env_and_pilot_outputs.zip` (or the printed outputs of cells 2, 3, 5, 6, 7, 9). Only after these are
checked does Phase III continue with `02_baseline_kaggle.ipynb` (Prompts 09–10).